# Book N-Gram Feature Extractor
Extracts the top 100 unigrams, bigrams, and trigrams from each book `.txt` file.
Used for genre classification feature engineering.

In [73]:
import re
import os
import pandas as pd
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Configuration
Edit the `BOOKS` dictionary to match your `.txt` filenames and genres.
Place all `.txt` files in the same folder as this notebook (or update `BOOKS_DIR`).

In [ ]:
# --- Configuration ---
BOOKS_DIR = "KnownBooks"   # Folder containing the .txt files
EXTRACT_N = 500            # Top n-grams to extract per book (candidates for genre aggregation)
TOP_N     = 100            # Top n-grams to keep per genre (final vocabulary)

BOOKS = {
    # biography (5 books)
    "AutobiographyOfBenjaminFranklin":  "biography",
    "LifeOfKingEdwardVII":              "biography",
    "NarrativeOfFrederickDouglass":     "biography",
    "StoryOfMyLife":                    "biography",
    "AutobiographyOfCharlesDarwin":     "biography",
    # fantasy (5 books)
    "TheWonderfulWizardOfOz":           "fantasy",
    "AliceInWonderland":                "fantasy",
    "ThroughTheLookingGlass":           "fantasy",
    "GrimmsFairyTales":                 "fantasy",
    "GulliversTravels":                 "fantasy",
    # horror (5 books)
    "Frankenstein":                     "horror",
    "CallOfCthulu":                     "horror",
    "Dracula":                          "horror",
    "TalesOfPoe":                       "horror",
    "DrJekyllAndMrHyde":                "horror",
    # romance (5 books)
    "PrideAndPrejudice":                "romance",
    "RomeoAndJuliet":                   "romance",
    "JaneEyre":                         "romance",
    "WutheringHeights":                 "romance",
    "SenseAndSensibility":              "romance",
    # sci-fi (5 books)
    "OnTheTrailOfTheSpacePirates":      "sci-fi",
    "PlagueShip":                       "sci-fi",
    "TheWarOfTheWorlds":                "sci-fi",
    "TheTimeMachine":                   "sci-fi",
    "TwentyThousandLeagues":            "sci-fi",
}

## Clean Raw Text Files (Project Gutenberg)
Strips the standard Project Gutenberg header and footer boilerplate from each `.txt` file
and saves cleaned versions as `<BookName>_clean.txt` in the same directory.
Run this once before extracting n-grams.

In [75]:
import re
import os

# Project Gutenberg delimiter patterns
START_PATTERN = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
END_PATTERN   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def clean_gutenberg(filename):
    """
    Strip Project Gutenberg header and footer from a .txt file.
    Saves the cleaned text as <filename>_clean.txt.
    Returns the cleaned text as a string.
    """
    path = os.path.join(BOOKS_DIR, filename + '.txt')
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # Find the start marker
    start_match = START_PATTERN.search(raw)
    if start_match:
        text = raw[start_match.end():]
    else:
        print(f'  [WARNING] No START marker found in {filename} — using full text')
        text = raw

    # Find the end marker
    end_match = END_PATTERN.search(text)
    if end_match:
        text = text[:end_match.start()]
    else:
        print(f'  [WARNING] No END marker found in {filename} — keeping text until EOF')

    text = text.strip()

    # Save cleaned version
    out_path = os.path.join(BOOKS_DIR, filename + '_clean.txt')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)

    original_words = len(raw.split())
    cleaned_words  = len(text.split())
    removed_words  = original_words - cleaned_words
    print(f'  {filename}: {original_words:,} → {cleaned_words:,} words  (removed {removed_words:,} boilerplate words)')
    return text

print('Cleaning Project Gutenberg boilerplate...\n')
for book in BOOKS:
    clean_gutenberg(book)

print('\nCleaned files saved as <BookName>_clean.txt')

Cleaning Project Gutenberg boilerplate...

  AutobiographyOfBenjaminFranklin: 79,264 → 76,203 words  (removed 3,061 boilerplate words)
  LifeOfKingEdwardVII: 150,777 → 147,715 words  (removed 3,062 boilerplate words)
  NarrativeOfFrederickDouglass: 43,819 → 40,750 words  (removed 3,069 boilerplate words)
  StoryOfMyLife: 137,927 → 134,873 words  (removed 3,054 boilerplate words)
  AutobiographyOfCharlesDarwin: 25,733 → 22,684 words  (removed 3,049 boilerplate words)
  TheWonderfulWizardOfOz: 42,692 → 39,649 words  (removed 3,043 boilerplate words)
  AliceInWonderland: 29,569 → 26,525 words  (removed 3,044 boilerplate words)
  ThroughTheLookingGlass: 32,789 → 29,752 words  (removed 3,037 boilerplate words)
  GrimmsFairyTales: 104,156 → 101,111 words  (removed 3,045 boilerplate words)
  GulliversTravels: 108,140 → 105,079 words  (removed 3,061 boilerplate words)
  Frankenstein: 78,106 → 75,042 words  (removed 3,064 boilerplate words)
  CallOfCthulu: 15,029 → 11,968 words  (removed 3,061 

## Helper Functions

In [76]:
STOP_WORDS = set(stopwords.words('english'))

def load_text(filename):
    """Read a .txt file and return its contents as a string."""
    path = os.path.join(BOOKS_DIR, filename + "_clean.txt")
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def tokenize(text):
    """Lowercase, strip punctuation, remove stop words."""
    words = re.findall(r'[a-z]+', text.lower())
    return [w for w in words if w not in STOP_WORDS and len(w) > 1]

def top_ngrams(tokens, n, top_n=TOP_N):
    """Return the top_n most common n-grams as a list of (ngram_string, count) tuples."""
    counts = Counter(ngrams(tokens, n))
    return [(' '.join(gram), count) for gram, count in counts.most_common(top_n)]

## Extract N-Grams for All Books

In [77]:
results = {}  # { book_name: { 'genre': ..., 'unigrams': [...], 'bigrams': [...], 'trigrams': [...] } }

for book, genre in BOOKS.items():
    print(f"Processing: {book} ({genre})...")
    text   = load_text(book)
    tokens = tokenize(text)

    results[book] = {
        'genre':    genre,
        'unigrams': top_ngrams(tokens, 1, top_n=EXTRACT_N),
        'bigrams':  top_ngrams(tokens, 2, top_n=EXTRACT_N),
        'trigrams': top_ngrams(tokens, 3, top_n=EXTRACT_N),
    }

print("\nDone!")

Processing: AutobiographyOfBenjaminFranklin (biography)...
Processing: LifeOfKingEdwardVII (biography)...
Processing: NarrativeOfFrederickDouglass (biography)...
Processing: StoryOfMyLife (biography)...
Processing: AutobiographyOfCharlesDarwin (biography)...
Processing: TheWonderfulWizardOfOz (fantasy)...
Processing: AliceInWonderland (fantasy)...
Processing: ThroughTheLookingGlass (fantasy)...
Processing: GrimmsFairyTales (fantasy)...
Processing: GulliversTravels (fantasy)...
Processing: Frankenstein (horror)...
Processing: CallOfCthulu (horror)...
Processing: Dracula (horror)...
Processing: TalesOfPoe (horror)...
Processing: DrJekyllAndMrHyde (horror)...
Processing: PrideAndPrejudice (romance)...
Processing: RomeoAndJuliet (romance)...
Processing: JaneEyre (romance)...
Processing: WutheringHeights (romance)...
Processing: SenseAndSensibility (romance)...
Processing: OnTheTrailOfTheSpacePirates (sci-fi)...
Processing: PlagueShip (sci-fi)...
Processing: TheWarOfTheWorlds (sci-fi)...
Pr

## Character Name Filtering
Uses NLTK's curated first-name corpus (~7,900 names) to remove any n-gram
whose tokens match a common English given name. This prevents the model from
memorising character names (e.g. *dorothy*, *darcy*, *victor*) instead of
learning genre signals.

In [78]:
nltk.download('names', quiet=True)
from nltk.corpus import names as nltk_names

# Curated list of ~7,900 common English given names (male + female)
PERSON_NAMES = set(n.lower() for n in nltk_names.words())
print(f"Loaded {len(PERSON_NAMES):,} common first names for filtering.")

removal_log = {}

print("\nFiltering n-grams that contain character first names...\n")
for book in BOOKS:
    removed = {'unigrams': [], 'bigrams': [], 'trigrams': []}

    for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
        kept = []
        for ngram, count in results[book][ngram_type]:
            if any(t in PERSON_NAMES for t in ngram.split()):
                removed[ngram_type].append((ngram, count))
            else:
                kept.append((ngram, count))
        results[book][ngram_type] = kept

    removal_log[book] = removed
    n_removed  = sum(len(removed[k]) for k in removed)
    names_hit  = sorted({t for k in removed
                           for ngram, _ in removed[k]
                           for t in ngram.split() if t in PERSON_NAMES})
    print(f"  {book}")
    print(f"    Names matched : {names_hit}")
    print(f"    N-grams removed: {n_removed}  "
          f"({len(removed['unigrams'])} uni / {len(removed['bigrams'])} bi / {len(removed['trigrams'])} tri)\n")

print("Filtering complete — results dict updated.")

Loaded 7,576 common first names for filtering.

Filtering n-grams that contain character first names...

  AutobiographyOfBenjaminFranklin
    Names matched : ['abraham', 'benjamin', 'bishop', 'boyd', 'bradford', 'brian', 'carry', 'case', 'chandler', 'charles', 'christ', 'coleman', 'collins', 'david', 'denny', 'drew', 'ford', 'fortune', 'france', 'frank', 'franklin', 'french', 'george', 'gilbert', 'godfrey', 'grace', 'grant', 'hamilton', 'harry', 'henry', 'hope', 'james', 'jefferson', 'john', 'june', 'king', 'louis', 'mark', 'may', 'miles', 'modesty', 'morris', 'norris', 'page', 'paul', 'penn', 'peter', 'quincy', 'quinn', 'ralph', 'richard', 'robert', 'rose', 'royal', 'samuel', 'saw', 'say', 'see', 'shirley', 'smith', 'son', 'sterling', 'temple', 'thomas', 'town', 'vi', 'way', 'webb', 'west', 'william', 'wilson', 'woodrow', 'wyndham']
    N-grams removed: 247  (27 uni / 77 bi / 143 tri)

  LifeOfKingEdwardVII
    Names matched : ['abbey', 'albert', 'alexandra', 'alfred', 'alice', 'apri

In [79]:
# ── Removal documentation: show exactly which n-grams were dropped per book ──
for book in BOOKS:
    log = removal_log[book]
    genre = BOOKS[book]
    total = sum(len(log[k]) for k in ['unigrams', 'bigrams', 'trigrams'])

    print(f"\n{'='*60}")
    print(f"  {book}  [{genre.upper()}]  —  {total} n-gram(s) removed")
    print(f"{'='*60}")

    for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
        items = log[ngram_type]
        if items:
            df = pd.DataFrame(items, columns=['ngram', 'count'])
            df.index += 1
            print(f"\n  Removed {ngram_type} ({len(items)}):")
            display(df)
        else:
            print(f"\n  No {ngram_type} removed.")



  AutobiographyOfBenjaminFranklin  [BIOGRAPHY]  —  247 n-gram(s) removed

  Removed unigrams (27):


,ngram,count
1,franklin,132
2,may,107
3,see,69
4,way,49
5,saw,37
6,town,34
7,son,31
8,king,30
9,william,29
10,richard,28



  Removed bigrams (77):


,ngram,count
1,poor richard,25
2,benjamin franklin,11
3,richard says,11
4,richard almanac,9
5,royal society,7
...,...,...
73,court louis,2
74,louis xvi,2
75,put king,2
76,king prize,2



  Removed trigrams (143):


...

,ngram,count
1,ned,324
2,see,151
3,saw,125
4,miles,109
5,say,76
6,may,72
7,rose,54
8,red,52
9,abraham,46
10,lincoln,46



  Removed bigrams (57):


,ngram,count
1,ned land,196
2,abraham lincoln,46
3,said ned,31
4,red sea,26
5,could see,24
6,friend ned,22
7,miles hour,18
8,ned conseil,17
9,replied ned,14
10,two miles,14



  Removed trigrams (101):


,ngram,count
1,said ned land,19
2,ned land conseil,15
3,replied ned land,11
4,sir said ned,7
5,cape good hope,7
...,...,...
97,ned land carried,2
98,good hope cape,2
99,ned land would,2
100,platform ned land,2


## Display Results per Book

In [80]:
def show_book(book_name):
    """Pretty-print the top n-grams for a single book."""
    data = results[book_name]
    print(f"\n{'='*60}")
    print(f"  {book_name}  [{data['genre'].upper()}]")
    print(f"{'='*60}")

    for label, key in [("TOP 100 UNIGRAMS", 'unigrams'),
                       ("TOP 100 BIGRAMS",  'bigrams'),
                       ("TOP 100 TRIGRAMS", 'trigrams')]:
        print(f"\n--- {label} ---")
        df = pd.DataFrame(data[key], columns=['ngram', 'count'])
        df.index += 1
        display(df)

# Show all books
for book in BOOKS:
    show_book(book)


  AutobiographyOfBenjaminFranklin  [BIOGRAPHY]

--- TOP 100 UNIGRAMS ---


,ngram,count
1,one,294
2,time,201
3,would,199
4,great,174
5,good,160
...,...,...
469,side,14
470,happened,14
471,attend,14
472,terms,14



--- TOP 100 BIGRAMS ---


,ngram,count
1,new york,38
2,printing house,28
3,new england,20
4,good deal,18
5,thousand pounds,18
...,...,...
419,ah says,2
420,says take,2
421,take kings,2
422,kings america,2



--- TOP 100 TRIGRAMS ---


...

,ngram,count
1,captain,644
2,nautilus,525
3,nemo,388
4,one,370
5,sea,357
...,...,...
473,small,20
474,slight,20
475,considerable,20
476,cause,20



--- TOP 100 BIGRAMS ---


,ngram,count
1,captain nemo,387
2,said conseil,53
3,said captain,45
4,let us,34
5,sir said,33
...,...,...
439,us captain,4
440,nets brought,4
441,beautiful specimens,4
442,lying side,4



--- TOP 100 TRIGRAMS ---


,ngram,count
1,said captain nemo,25
2,next day th,14
3,replied captain nemo,13
4,captain nemo went,9
5,sir said captain,9
...,...,...
395,captain nemo conducted,2
396,turned captain nemo,2
397,captain nemo hand,2
398,last words escaped,2


## Genre-Level N-Gram Aggregation
Pools n-gram counts across all books within each genre, then takes the top `TOP_N`
per genre. A word that appears consistently across multiple books in a genre ranks
higher than one that dominates a single book — giving a more representative vocabulary.

In [81]:
from collections import defaultdict

# Sum n-gram counts across all books within the same genre
genre_counts = defaultdict(lambda: {t: Counter() for t in ['unigrams', 'bigrams', 'trigrams']})

for book, data in results.items():
    genre = data['genre']
    for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
        for ngram, count in data[ngram_type]:
            genre_counts[genre][ngram_type][ngram] += count

# Take top-TOP_N per genre per n-gram type
genre_top = {}
for genre in sorted(genre_counts):
    genre_top[genre] = {
        ngram_type: genre_counts[genre][ngram_type].most_common(TOP_N)
        for ngram_type in ['unigrams', 'bigrams', 'trigrams']
    }

print(f"Genre-level top-{TOP_N} n-grams computed:\n")
for genre in sorted(genre_top):
    u = len(genre_top[genre]['unigrams'])
    b = len(genre_top[genre]['bigrams'])
    t = len(genre_top[genre]['trigrams'])
    print(f"  {genre:12}: {u} unigrams, {b} bigrams, {t} trigrams")

total = sum(len(genre_top[g][t]) for g in genre_top for t in ['unigrams', 'bigrams', 'trigrams'])
print(f"\nTotal vocabulary candidates: {total} (before cross-genre deduplication)")

Genre-level top-100 n-grams computed:

  biography   : 100 unigrams, 100 bigrams, 100 trigrams
  fantasy     : 100 unigrams, 100 bigrams, 100 trigrams
  horror      : 100 unigrams, 100 bigrams, 100 trigrams
  romance     : 100 unigrams, 100 bigrams, 100 trigrams
  sci-fi      : 100 unigrams, 100 bigrams, 100 trigrams

Total vocabulary candidates: 1500 (before cross-genre deduplication)


## Top 10 Unigrams per Genre
Aggregated across all books in each genre — shows the most genre-representative words.

In [82]:
comparison = {
    genre: [ngram for ngram, _ in genre_top[genre]['unigrams'][:10]]
    for genre in sorted(genre_top)
}

comp_df = pd.DataFrame(comparison)
comp_df.index = [f"#{i+1}" for i in range(10)]
print("Top 10 unigrams per genre (aggregated across books, stop words removed):\n")
display(comp_df)

Top 10 unigrams per genre (aggregated across books, stop words removed):



,biography,fantasy,horror,romance,sci-fi
#1,one,said,one,would,one
#2,great,one,said,could,said
#3,little,could,could,mr,captain
#4,time,would,would,said,could
#5,would,little,upon,one,would
#6,mr,came,us,mrs,time
#7,many,went,must,must,two
#8,made,upon,man,well,us
#9,much,great,time,much,strong
#10,could,time,shall,miss,man


## Export Genre N-Grams to CSV
One CSV per n-gram type, one row per genre-level top-`TOP_N` entry.

In [83]:
for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
    rows = []
    for genre in sorted(genre_top):
        for rank, (ngram, count) in enumerate(genre_top[genre][ngram_type], start=1):
            rows.append({
                'genre': genre,
                'rank':  rank,
                'ngram': ngram,
                'count': count,
            })
    df = pd.DataFrame(rows)
    out_path = f"ngrams/top100_{ngram_type}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

Saved: ngrams/top100_unigrams.csv
Saved: ngrams/top100_bigrams.csv
Saved: ngrams/top100_trigrams.csv


## Export Vocabulary for Classifier
Saves `vocabulary.csv` — the union of genre-level top-`TOP_N` n-grams across all five genres,
de-duplicated. The classifier notebook uses this list to build its TF-IDF feature matrix.

In [84]:
# Union of genre-level top-100 n-grams across all genres
all_ngrams = set()
for data in genre_top.values():
    for word,   _ in data['unigrams']:  all_ngrams.add(word)
    for phrase, _ in data['bigrams']:   all_ngrams.add(phrase)
    for phrase, _ in data['trigrams']:  all_ngrams.add(phrase)

n_uni = len({w for data in genre_top.values() for w, _ in data['unigrams']})
n_bi  = len({p for data in genre_top.values() for p, _ in data['bigrams']})
n_tri = len({p for data in genre_top.values() for p, _ in data['trigrams']})

vocab_df = pd.DataFrame(sorted(all_ngrams), columns=['word'])
vocab_df.to_csv('FeatureTrainingData/vocabulary.csv', index=False)
print(f"Vocabulary size: {len(vocab_df)} entries  ({n_uni} unigrams + {n_bi} bigrams + {n_tri} trigrams)")
print(f"Saved to 'FeatureTrainingData/vocabulary.csv'")

Vocabulary size: 1109 entries  (214 unigrams + 399 bigrams + 496 trigrams)
Saved to 'FeatureTrainingData/vocabulary.csv'


## Build Feature Vector CSV
Applies TF-IDF (restricted to the vocabulary) to every labelled text partition
and saves the result as a ready-to-train feature matrix.

**Requires:** `TestingData/partitions.csv` from `Book partitioner.ipynb`

**Outputs:**
- `FeatureTrainingData/featurevector.csv` — one row per partition, columns for `partition_id`, `genre`, and one TF-IDF score per vocabulary entry (unigrams + bigrams + trigrams)
- `FeatureTrainingData/vectorizer.pkl` — fitted vectorizer for transforming new unseen text

In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Load labelled partitions (produced by Book partitioner.ipynb)
partitions = pd.read_csv('TestingData/partitions.csv')

# Full vocabulary: unigrams + bigrams + trigrams in sorted order (same as vocabulary.csv)
vocab_list = sorted(all_ngrams)

# Fit TF-IDF restricted to the shared vocabulary.
# ngram_range=(1,3) enables phrase matching; stop_words removes stopwords before
# generating n-grams so phrases match the stopword-filtered extraction logic.
vectorizer = TfidfVectorizer(
    vocabulary    = vocab_list,
    ngram_range   = (1, 3),
    stop_words    = 'english',
    sublinear_tf  = True,
    strip_accents = 'unicode',
    analyzer      = 'word',
    token_pattern = r'[a-z]+',
    lowercase     = True,
)

X = vectorizer.fit_transform(partitions['text'])
feature_names = vectorizer.get_feature_names_out()

# Assemble: partition_id | genre | feature_1 | feature_2 | ...
feat_df = pd.DataFrame(X.toarray(), columns=feature_names)
feat_df.insert(0, 'genre',        partitions['genre'].values)
feat_df.insert(0, 'partition_id', partitions['partition_id'].values)

feat_df.to_csv('FeatureTrainingData/featurevector.csv', index=False)
joblib.dump(vectorizer, 'FeatureTrainingData/vectorizer.pkl')

print(f"Saved: FeatureTrainingData/featurevector.csv")
print(f"  {feat_df.shape[0]} partitions  ×  {len(feature_names)} features")
print(f"  ({n_uni} unigram + {n_bi} bigram + {n_tri} trigram features)")
print(f"Saved: FeatureTrainingData/vectorizer.pkl")
feat_df.iloc[:3, :8]   # preview first few columns


Saved: FeatureTrainingData/featurevector.csv
  5000 partitions  ×  1109 features
  (214 unigram + 399 bigram + 496 trigram features)
Saved: FeatureTrainingData/vectorizer.pkl


,partition_id,genre,accompanied princess,accompanied princess wales,across,actions man man,ad le,ad le came
0,AutobiographyOfBenjaminFranklin_partition_a,biography,0.0,0.0,0.0,0.0,0.0,0.0
1,AutobiographyOfBenjaminFranklin_partition_b,biography,0.0,0.0,0.0,0.0,0.0,0.0
2,AutobiographyOfBenjaminFranklin_partition_c,biography,0.0,0.0,0.0,0.0,0.0,0.0
